Mục tiêu: `Raw dataset -> validation`

Kiểm tra:

```
Dataset Exists
Dataset Size
Image Shape
Label Range
Pixel Range
Missing Samples
Corrupted Samples
```

Sinh:
```
reports/

validation_report.md
validation.json
```

In [1]:
from pathlib import Path
import json

import numpy as np

from torchvision.datasets import MNIST

In [2]:
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "raw"
REPORT_DIR = PROJECT_ROOT / "reports"

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MNIST_DIR = RAW_DIR / "mnist"

print(MNIST_DIR)

D:\Nam 3\Neural Network\Project CNN MNIST\data\raw\mnist


In [3]:
manifest_path = RAW_DIR / "ingestion_manifest.json"

if not manifest_path.exists():
    raise FileNotFoundError(
        "Run 01_data_ingestion.ipynb first."
    )

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)

manifest

{'dataset': 'MNIST',
 'status': 'downloaded',
 'train_samples': 60000,
 'test_samples': 10000}

In [4]:
train_dataset = MNIST(
    root=str(MNIST_DIR),
    train=True,
    download=False
)

test_dataset = MNIST(
    root=str(MNIST_DIR),
    train=False,
    download=False
)

In [5]:
validation = {
    "dataset_exists": True,
    "sample_count_valid": True,
    "shape_valid": True,
    "label_valid": True,
    "pixel_range_valid": True,
    "missing_samples": 0,
    "corrupted_samples": 0
}

In [6]:
expected_train = manifest["train_samples"]
expected_test = manifest["test_samples"]

if len(train_dataset) != expected_train:
    validation["sample_count_valid"] = False

if len(test_dataset) != expected_test:
    validation["sample_count_valid"] = False

In [7]:
for idx in range(len(train_dataset)):

    image, _ = train_dataset[idx]

    image_np = np.array(image)

    if image_np.shape != (28, 28):

        validation["shape_valid"] = False

        break

In [8]:
valid_labels = set(range(10))

for label in train_dataset.targets.numpy():

    if int(label) not in valid_labels:

        validation["label_valid"] = False

        break

In [9]:
for idx in range(len(train_dataset)):

    image, _ = train_dataset[idx]

    image_np = np.array(image)

    if image_np.min() < 0:

        validation["pixel_range_valid"] = False

        break

    if image_np.max() > 255:

        validation["pixel_range_valid"] = False

        break

In [10]:
for idx in range(len(train_dataset)):

    try:

        image, _ = train_dataset[idx]

        image_np = np.array(image)

        _ = image_np.shape

    except Exception:

        validation["corrupted_samples"] += 1

In [11]:
for idx in range(len(train_dataset)):

    image, label = train_dataset[idx]

    if image is None:

        validation["missing_samples"] += 1

In [12]:
validation["overall_status"] = (
    validation["sample_count_valid"]
    and validation["shape_valid"]
    and validation["label_valid"]
    and validation["pixel_range_valid"]
    and validation["missing_samples"] == 0
    and validation["corrupted_samples"] == 0
)

validation

{'dataset_exists': True,
 'sample_count_valid': True,
 'shape_valid': True,
 'label_valid': True,
 'pixel_range_valid': True,
 'missing_samples': 0,
 'corrupted_samples': 0,
 'overall_status': True}

In [13]:
json_report_path = (
    REPORT_DIR /
    "validation.json"
)

with open(
    json_report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        validation,
        f,
        indent=4
    )

print(json_report_path)

D:\Nam 3\Neural Network\Project CNN MNIST\data\reports\validation.json


In [14]:
report_md = f"""
# Dataset Validation Report

## Dataset

MNIST

---

## Validation Results

| Check | Result |
|---------|---------|
| Dataset Exists | {validation['dataset_exists']} |
| Sample Count | {validation['sample_count_valid']} |
| Shape Validation | {validation['shape_valid']} |
| Label Validation | {validation['label_valid']} |
| Pixel Validation | {validation['pixel_range_valid']} |
| Missing Samples | {validation['missing_samples']} |
| Corrupted Samples | {validation['corrupted_samples']} |

---

## Final Status

PASS = {validation['overall_status']}
"""

In [15]:
md_report_path = (
    REPORT_DIR /
    "validation_report.md"
)

with open(
    md_report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(report_md)

print(md_report_path)

D:\Nam 3\Neural Network\Project CNN MNIST\data\reports\validation_report.md
